### CSU CM 515 (Tai Montgomery, 2/23/2026)
# 📄 File Formats
In this notebook, we'll explore several file types commonly encountered in bioinformatics.

### 🧩 Exercise 1: Viewing `fastq` files.
Here we'll examine the contents of a `fastq` file.

1.1. Download `fastq` files from a paired-end sequencing run from E. coli: [fastq files](https://drive.google.com/drive/folders/1L7vRFF47Ab-kMINPHC9GRz0x6N0aipLb?usp=share_link).  If you're working on the server, you already have these files.

1.2. From your terminal, use the Linux/Unix command `less` to view the contents of these files one screenful at a time (`zless` can be used for `.gz` files).  Ignore the `%%bash` line and do not include it in your code!

In [ ]:
%%bash
less SRR292770_1.fastq

`less` has several useful shortcuts:
 * `=`: line count
 * `-N`: show line numbers
 * `g`: go to first line
 * `G`: go to last line
 * `\pattern`: go to nex line matching pattern
 * `q`: quit

1.3. What information is contained in each line?  Determine the length of reads using the Linux/Unix command `wc`. First you have to `echo` the sequence and then pipe (`|`) the output to `wc`:

In [ ]:
%%bash
echo sequence_line | wc

1.4. How could you determine the number of reads in these files?  Try using `wc file_name`.

### 🧩 Exercise 2: Preprocessing `fastq` files with `fastp`.
Here we'll trim adapter sequences and low quality bases using `fastp`. See the [GitHub repository](https://github.com/OpenGene/fastp) for information.

To install `fastp`, we'll use the package management software `conda`. If you need to first install `conda`, follow the instructions below.

#### Install `conda` (Miniconda)
Download the appropriate installer for your computer from this link and then install the package:
https://www.anaconda.com/download/success




#### Install `fastp` using `conda` within an environment called `cm515`:

In [ ]:
%%bash
conda create -n cm515 -c conda-forge -c bioconda --strict-channel-priority fastp

Activate the `cm515` environment:

In [ ]:
%%bash
conda activate cm515

#### Run `fastp`

2.1. Replace the file names and run the following code for `fastp` from your terminal.

In [ ]:
%%bash
fastp -i 'fastq_file_1' -I 'fastq_file_2' \
-o 'trimmed_fastq_file_1' -O 'trimmed_fastq_file_2' \
-h report.html

This code uses default parameters trimming adapters and filtering reads. See the [GitHub repository](https://github.com/OpenGene/fastp) for information on how to modify these parameters.

2.2. Using the results printed to the terminal and the quality report, `report.html`, answer the following questions:
 * How long on average are the reads pre- and post-filtering?
 * How does the quality change across the length of the reads?
 * How does the nucelotide distribution change across the length of the reads?

2.3. View the trimmed `fastq` files with `less`.  What changed from the original files?

### 🧩 Exercise 3: Aligning reads with `bowtie2`.
Here we'll align reads in the trimmed `fastq` files to the E. coli genome. For information about `bowtie2`, see the [bowtie2 manual](http://bowtie-bio.sourceforge.net/bowtie2/index.shtml).

Before running `bowtie2`, we'll need to install it with `conda` within the `cm515` environment:

In [ ]:
%%bash
conda install -c bioconda -c conda-forge bowtie2

#### Run `bowtie2`

3.1. The first step in doing a `bowtie` alignment is to obtain or create a `bowtie index`. Edit the text in quotes below and then paste it into the terminal to run it. For `index_prefix`, use `ecoli`. The genome sequence file can be downloaded from [Google Drive](https://drive.google.com/drive/folders/1L7vRFF47Ab-kMINPHC9GRz0x6N0aipLb?usp=share_link).

In [ ]:
%%bash
bowtie2-build 'genome_sequence_file' 'index_prefix'

3.2. Once we have an index (it only has to be generated once for a specific genome), we can do the alignment. Edit the quoted text below and then paste it into the terminal to run it. The trimmed `fastq` files were generated by `fastp` above. Then number of threads will depend on your comptuter. By default, use 8.

In [ ]:
bowtie2 -p 'number_of_threads' -x 'index_prefix' \
-1 'trimmed_fastq_file_1' -2 'trimmed_fastq_file_2'  \
-S output_file.sam

3.3. When the run is complete, answer the following questions:
 * What information does `bowtie` provide about the alignment?
 * What proportion of reads aligned to the reference?
 * What files were generated by `bowtie2`?

### 🧩 Exercise 4: Parsing `sam` files with Python.
Here we'll use python to filter for only reads that matched perfectly to the genome.

4.1. Use the `less` command to view the contents of the `sam` file generated using `bowtie2` above.  Which are the header lines? What information is in the various fields?

4.2. Many alignment programs use the optional 12th field of the sam file to indicate mismatches to the genome. Perfectly mathing reads have the tag `NM:i:0`, which refers to an edit distance of 0. It's easy to parse these reads using grep or python.  Here we'll use Python.

In [ ]:
import sys

in_sam = sys.argv[1]
out_sam = sys.argv[2]

with open(in_sam) as fin, open(out_sam, "w") as fout:
    for line in fin:
        # write header lines to file
        if line.startswith("@"):
            fout.write(line)
            continue

        # write only perfectly matching reads to the output file
        if "NM:i:0" in line:
            fout.write(line)


4.3. Use the `less` command to view the output.  How do you know the script did what it was supposed to? Next time we meet we'll compare these results to the original alignment results.